# PyTorch Text Classification

## Setup

In [1]:
# Set up Kaggle API credentials for downloading datasets
import os  # For file and directory operations
import shutil  # For file copying

kaggle_json_path = './kaggle.json'  # Path to your Kaggle API key
kaggle_dir = os.path.expanduser('~/.kaggle')  # Default Kaggle directory
os.makedirs(kaggle_dir, exist_ok=True)  # Create directory if it doesn't exist
shutil.copy(kaggle_json_path, os.path.join(kaggle_dir, 'kaggle.json'))  # Copy API key
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)  # Set permissions
print('Kaggle API key set up successfully.')

Kaggle API key set up successfully.


In [2]:
# Install and import opendatasets, then download the bean leaf lesions dataset from Kaggle
!pip install -q opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection")

Skipping, found downloaded files in ".\news-headlines-dataset-for-sarcasm-detection" (use force=True to force download)


## Imports

In [ ]:
# Third-party libraries
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import pandas as pd
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import numpy as np
from sklearn.model_selection import train_test_split

# Reproducibility
torch.manual_seed(42)

# Set device to GPU if available, else CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch version: {torch.__version__}")
print(f"Is ROCm/CUDA available: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

c:\Users\lovep\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Image data classification with PyTorch

In [ ]:
data_df = pd.read_json('news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json',lines=True)
data_df.dropna(inplace=True)
data_df.drop_duplicates(inplace=True)
data_df.drop(columns=['article_link'], inplace=True)
print(data_df.shape)
data_df.head()

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(np.array(data_df['headline']), np.array(data_df['is_sarcastic']), test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=0.5, random_state=42)

print(f"Training set size: {len(X_train)} (% {len(X_train)/len(data_df)*100:.2f}%)")
print(f"Validation set size: {len(X_val)} (% {len(X_val)/len(data_df)*100:.2f}%)")
print(f"Test set size: {len(X_test)} (% {len(X_test)/len(data_df)*100:.2f}%)")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

In [ ]:
class dataset(Dataset):
    def __init__(self,X,Y):
        self.X = [tokenizer(x, 
                            padding='max_length', 
                            truncation=True, 
                            max_length=100, 
                            return_tensors='pt').to(device) 
                            for x in X
        ]
        self.Y = torch.Tensor(Y, dtype=torch.float32).to(device)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
    
training_data = dataset(X_train, Y_train)
validation_data = dataset(X_val, Y_val)
test_data = dataset(X_test, Y_test)